In [1]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# 导入库，创建输出目录
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

Path('results/rfe').mkdir(parents=True, exist_ok=True)


# =============================================================================
# 加载数据、CV设置及最优超参数
# =============================================================================

df = pd.read_csv('data/full_dataset.csv')

with open('data/cv_setup.pkl', 'rb') as f:
    cv_setup = pickle.load(f)

feature_cols = cv_setup['feature_cols']
target_col   = cv_setup['target_col']
outer_folds  = cv_setup['outer_folds']

X = df[feature_cols].values
y = df[target_col].values

with open('models/rf/rf_results.json', 'r') as f:
    rf_params = json.load(f)['final_model']['params']


In [2]:
# =============================================================================
# 结构级别交叉验证评估函数
# =============================================================================

def structure_level_cv_evaluate(X_full, y_full, feature_indices, outer_folds, rf_params):
    """
    结构级别5折交叉验证评估模型性能。
    每折内部独立标准化，避免数据泄露。
    返回各折的MAE、RMSE均值与标准差，以及平均特征重要性。
    """
    fold_rmses       = []
    fold_maes        = []
    fold_importances = []

    for outer_fold in outer_folds:
        train_idx = outer_fold['train_atoms_idx']
        test_idx  = outer_fold['test_atoms_idx']

        X_train = X_full[train_idx][:, feature_indices]
        X_test  = X_full[test_idx][:, feature_indices]
        y_train = y_full[train_idx]
        y_test  = y_full[test_idx]

        scaler          = StandardScaler()
        X_train_scaled  = scaler.fit_transform(X_train)
        X_test_scaled   = scaler.transform(X_test)

        rf = RandomForestRegressor(**rf_params, random_state=42, n_jobs=-1)
        rf.fit(X_train_scaled, y_train)

        y_pred = rf.predict(X_test_scaled)
        fold_rmses.append(np.sqrt(mean_squared_error(y_test, y_pred)))
        fold_maes.append(mean_absolute_error(y_test, y_pred))
        fold_importances.append(rf.feature_importances_)

    return {
        'rmse_mean':        float(np.mean(fold_rmses)),
        'rmse_std':         float(np.std(fold_rmses)),
        'mae_mean':         float(np.mean(fold_maes)),
        'mae_std':          float(np.std(fold_maes)),
        'mean_importances': np.mean(fold_importances, axis=0)
    }




In [3]:
# =============================================================================
# 递归特征消除（每次删除最不重要的特征）
# =============================================================================

rfe_results = {
    'n_features':          [],
    'cv_rmse_mean':        [],
    'cv_rmse_std':         [],
    'cv_mae_mean':         [],
    'cv_mae_std':          [],
    'selected_features':   [],
    'removed_feature':     [],
    'feature_importances': []
}

available_features        = feature_cols.copy()
available_feature_indices = list(range(len(feature_cols)))

while len(available_features) >= 1:
    cv_result = structure_level_cv_evaluate(
        X, y, available_feature_indices, outer_folds, rf_params
    )

    importances = cv_result['mean_importances']

    rfe_results['n_features'].append(len(available_features))
    rfe_results['cv_rmse_mean'].append(cv_result['rmse_mean'])
    rfe_results['cv_rmse_std'].append(cv_result['rmse_std'])
    rfe_results['cv_mae_mean'].append(cv_result['mae_mean'])
    rfe_results['cv_mae_std'].append(cv_result['mae_std'])
    rfe_results['selected_features'].append(available_features.copy())
    rfe_results['feature_importances'].append(importances.tolist())

    if len(available_features) == 1:
        rfe_results['removed_feature'].append(None)
        break

    least_important_idx = np.argmin(importances)
    removed_feature     = available_features[least_important_idx]
    rfe_results['removed_feature'].append(removed_feature)

    del available_features[least_important_idx]
    del available_feature_indices[least_important_idx]

# 确定最优特征数量（测试集MAE最小）
cv_mae_array  = np.array(rfe_results['cv_mae_mean'])
optimal_idx   = np.argmin(cv_mae_array)
optimal_n     = rfe_results['n_features'][optimal_idx]

selected_features = rfe_results['selected_features'][optimal_idx]
removed_features  = [f for f in feature_cols if f not in selected_features]

all_features_rmse     = rfe_results['cv_rmse_mean'][0]
all_features_rmse_std = rfe_results['cv_rmse_std'][0]
all_features_mae      = rfe_results['cv_mae_mean'][0]
all_features_mae_std  = rfe_results['cv_mae_std'][0]
optimal_rmse          = rfe_results['cv_rmse_mean'][optimal_idx]
optimal_rmse_std      = rfe_results['cv_rmse_std'][optimal_idx]
optimal_mae           = rfe_results['cv_mae_mean'][optimal_idx]
optimal_mae_std       = rfe_results['cv_mae_std'][optimal_idx]

relative_change_rmse = (optimal_rmse - all_features_rmse) / all_features_rmse * 100
relative_change_mae  = (optimal_mae  - all_features_mae)  / all_features_mae  * 100




In [4]:
# =============================================================================
# 保存结果
# =============================================================================

# 每一步评分
pd.DataFrame({
    'n_features':   rfe_results['n_features'],
    'cv_rmse_mean': rfe_results['cv_rmse_mean'],
    'cv_rmse_std':  rfe_results['cv_rmse_std'],
    'cv_mae_mean':  rfe_results['cv_mae_mean'],
    'cv_mae_std':   rfe_results['cv_mae_std']
}).to_csv('results/rfe/rfe_scores.csv', index=False)

# 每一步特征列表与重要性
with open('results/rfe/rfe_features.json', 'w') as f:
    json.dump({
        'n_features':          rfe_results['n_features'],
        'selected_features':   rfe_results['selected_features'],
        'removed_feature':     rfe_results['removed_feature'],
        'feature_importances': rfe_results['feature_importances']
    }, f, indent=4)

# 最优特征子集
with open('results/rfe/optimal_features.json', 'w') as f:
    json.dump({
        'original_n_features': len(feature_cols),
        'optimal_n_features':  optimal_n,
        'n_removed':           len(removed_features),
        'retention_rate':      optimal_n / len(feature_cols),
        'original_features':   feature_cols,
        'selected_features':   selected_features,
        'removed_features':    removed_features,
        'optimal_score': {
            'rmse_mean':               float(optimal_rmse),
            'rmse_std':                float(optimal_rmse_std),
            'mae_mean':                float(optimal_mae),
            'mae_std':                 float(optimal_mae_std),
            'relative_change_rmse_pct': float(relative_change_rmse),
            'relative_change_mae_pct':  float(relative_change_mae)
        },
        'all_features_score': {
            'rmse_mean': float(all_features_rmse),
            'rmse_std':  float(all_features_rmse_std),
            'mae_mean':  float(all_features_mae),
            'mae_std':   float(all_features_mae_std)
        }
    }, f, indent=4)

# 特征删除顺序
removal_order = [f for f in rfe_results['removed_feature'] if f is not None]
removal_order.reverse()
feature_ranking = [{'rank': 1, 'feature': rfe_results['selected_features'][-1][0], 'status': 'selected'}]
for rank, feat in enumerate(removal_order, 2):
    feature_ranking.append({
        'rank':    rank,
        'feature': feat,
        'status':  'selected' if feat in selected_features else 'removed'
    })
pd.DataFrame(feature_ranking).to_csv('results/rfe/feature_ranking.csv', index=False)